In [1]:
!pip install -U sentencepiece transformers soundfile datasets --q

In [2]:
import gc
import warnings
warnings.filterwarnings("ignore")

import torch
import chess

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

torch.manual_seed(42)
clear_memory()

In [ ]:
from huggingface_hub import login

HF_TOKEN = ""
if HF_TOKEN:
    try:
        login(token=HF_TOKEN)
        print("HF login successful.")
    except Exception as e:
        print("HF login failed, proceeding without:", e)
else:
    print("No HF token provided. Proceeding without authentication.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login successful.


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "google/gemma-3-1b-it" 

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
print(f"Loading model (dtype={dtype})...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=None if dtype is torch.float32 else dtype,
    device_map="auto",
)
model.eval()
print("Model ready.")


Loading tokenizer...
Loading model (dtype=torch.bfloat16)...
Model ready.


In [5]:
def safe_generate(prompt: str, max_new_tokens: int = 56, temperature: float = 0.8) -> str:
    """
    Faster generation: concise outputs, strips prompt echo, handles OOM.
    """
    if not prompt.strip():
        return "[Invalid prompt]"

    device = next(model.parameters()).device
    try:
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512  # keep prompt compact
        ).to(device)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                top_p=0.85,
                temperature=temperature,   # a bit warmer for human tone
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True
            )

        text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if prompt in text:
            text = text.split(prompt, 1)[-1]
        return text.strip() or "[Empty generation]"
    except RuntimeError as e:
        if "CUDA" in str(e):
            torch.cuda.empty_cache()
        return f"[RuntimeError: {str(e)[:80]}...]"
    except Exception as e:
        return f"[Error: {str(e)[:80]}...]"

# ---- text cleaners ----
def _strip_fences_and_labels(text: str) -> str:
    import re
    txt = re.sub(r"```.*?```", "", text, flags=re.DOTALL)
    txt = txt.replace("**Commentary:**", "").replace("Commentary:", "").strip()
    return txt

def clean_commentary(text: str, max_sentences: int = 2) -> str:
    """
    For move-by-move quips: trim to ≤ max_sentences (default 2).
    """
    import re
    txt = _strip_fences_and_labels(text)
    parts = [p.strip() for p in re.split(r'(?<=[.!?])\s+', txt) if len(p.strip()) > 6]
    if max_sentences is not None and len(parts) > max_sentences:
        parts = parts[:max_sentences]
    return " ".join(parts) if parts else "[No commentary]"

def clean_summary(text: str) -> str:
    """
    For post-game summaries: DO NOT cap sentences.
    """
    return _strip_fences_and_labels(text)

In [6]:
# Lightweight static evaluation: material + mobility + center control
PIECE_VALUE = {
    chess.PAWN: 1.0, chess.KNIGHT: 3.2, chess.BISHOP: 3.3,
    chess.ROOK: 5.0, chess.QUEEN: 9.0, chess.KING: 0.0
}
CENTER = {chess.D4, chess.E4, chess.D5, chess.E5}
EXT_CENTER = {
    chess.C3, chess.D3, chess.E3, chess.F3,
    chess.C4, chess.F4, chess.C5, chess.F5,
    chess.C6, chess.D6, chess.E6, chess.F6
}

def material_score(board: chess.Board) -> float:
    score = 0.0
    for pt in PIECE_VALUE:
        score += len(board.pieces(pt, chess.WHITE)) * PIECE_VALUE[pt]
        score -= len(board.pieces(pt, chess.BLACK)) * PIECE_VALUE[pt]
    return score

def mobility_score(board: chess.Board) -> float:
    # quick proxy; same both sides is fine for a simple heuristic
    return board.legal_moves.count() * 0.02

def center_control_score(board: chess.Board) -> float:
    score = 0.0
    for sq in CENTER | EXT_CENTER:
        piece = board.piece_at(sq)
        if piece:
            v = 0.25 if sq in CENTER else 0.1
            score += v if piece.color == chess.WHITE else -v
    return score

def eval_white(board: chess.Board) -> float:
    return material_score(board) + center_control_score(board) + mobility_score(board)

def classify_move(board_before: chess.Board, move: chess.Move) -> tuple[str, list[str]]:
    """
    Classify move quality (heuristic) and produce talking points.
    Returns: (label, [reasons])
    """
    reasons = []
    mover_is_white = board_before.turn

    score_before = eval_white(board_before)

    board_after = board_before.copy()
    board_after.push(move)
    score_after = eval_white(board_after)

    # delta from mover's perspective
    delta = (score_after - score_before) if mover_is_white else (score_before - score_after)

    # rough thresholds
    if delta >= 0.60:
        label = "Excellent"
    elif delta >= 0.20:
        label = "Good"
    elif delta >= -0.20:
        label = "Inaccuracy"
    elif delta >= -0.60:
        label = "Mistake"
    else:
        label = "Blunder"

    # tags
    if board_before.gives_check(move):
        reasons.append("gives check")
    if board_before.is_capture(move):
        reasons.append("wins material" if delta > 0.15 else "trades material")
    if board_after.is_check():
        reasons.append("creates mating threats" if delta > 0.4 else "keeps king under fire")
    if board_before.is_castling(move):
        reasons.append("improves king safety")

    # development / center hints
    to_sq = move.to_square
    if chess.square_file(to_sq) in (3, 4) or chess.square_rank(to_sq) in (3, 4):
        reasons.append("improves central control")
    piece = board_before.piece_at(move.from_square)
    if piece and piece.piece_type in (chess.KNIGHT, chess.BISHOP) and chess.square_rank(move.from_square) in (0, 7):
        reasons.append("develops a minor piece")

    return label, reasons

def humanize_commentary(san: str, label: str, reasons: list[str]) -> str:
    lead = {
        "Excellent": "Powerful idea.",
        "Good": "Solid and purposeful.",
        "Inaccuracy": "A bit soft.",
        "Mistake": "This misfires.",
        "Blunder": "That’s a blunder."
    }[label]

    if reasons:
        from random import sample
        r = ", ".join(sample(reasons, k=min(2, len(reasons))))
        tail = f" It {r}."
    else:
        tail = ""

    verdict = f"[{label}]"
    return f"{verdict} {lead}{tail}"


In [7]:
SYSTEM_STYLE = (
    "You are a concise chess commentator. In ≤2 sentences, explain the *idea* of the move (plans/tactics), "
    "not just restating the SAN."
)

DETAILED_SUMMARY_STYLE = (
    "You are a chess analyst summarizing a completed game. "
    "Write a detailed, instructive analysis (180–250 words) for a general chess audience. "
    "Include: 1) the opening family or variation based on first moves, "
    "2) key turning points and tactical ideas, 3) examples of good and bad moves, "
    "4) how momentum shifted, 5) endgame or final tactics that decided the result, "
    "6) overall lessons learned. Use clear, narrative prose, not bullet points."
)

def make_comment_prompt(fen: str, san: str, verdict_line: str, last_6: str) -> str:
    return (
        f"{SYSTEM_STYLE}\n\n"
        f"FEN: {fen}\n"
        f"Recent moves: {last_6 or 'None'}\n"
        f"Move: {san}\n"
        f"Coach note: {verdict_line}\n"
        f"Commentary:"
    )

def make_summary_prompt(result_str: str, reason: str, san_moves: list[str]) -> str:
    first_8 = " ".join(san_moves[:8])
    all_moves = " ".join(san_moves[-300:])
    return (
        f"{DETAILED_SUMMARY_STYLE}\n\n"
        f"Result: {result_str} ({reason}).\n"
        f"Opening moves: {first_8}\n"
        f"Full move list: {all_moves}\n\n"
        f"Detailed Summary:"
    )

def print_board(board: chess.Board):
    # White at bottom; compatible with older python-chess (no 'flipped' arg)
    print(board.unicode(borders=True, invert_color=False))

In [8]:
import re

PIECE_LETTERS = set("nbrqk")
DECORATION_RE = re.compile(r"[+#?!]+")

def sanitize_san(user_input: str) -> str:
    """
    Remove SAN decorations like +, #, !, ?, ?!, !! to avoid misleading parsing.
    """
    s = user_input.strip()
    s = DECORATION_RE.sub("", s)
    return s

def normalize_san(user_input: str) -> str:
    """
    Convert case-insensitive SAN to proper casing and sanitize decorations:
    - nf6 -> Nf6
    - o-o, 0-0 -> O-O ; o-o-o -> O-O-O
    - strips +, #, !, ?, ?!, !! etc.
    """
    s = sanitize_san(user_input)

    # Castling variants
    s_castle = s.replace("0-0-0", "O-O-O").replace("0-0", "O-O")
    s_castle = re.sub(r'(?i)o-o-o', "O-O-O", s_castle)
    s_castle = re.sub(r'(?i)o-o', "O-O", s_castle)
    if s_castle != s:
        return s_castle

    if s and s[0] in PIECE_LETTERS:
        s = s[0].upper() + s[1:]
    return s

In [9]:
def run_one_game():
    print("Chess Commentary Chatbot (one game)")
    board = chess.Board()
    move_history: list[str] = []

    # show starting board (White at bottom)
    print_board(board)

    while True:
        # Terminal result -> summarize and exit (trusted board state)
        if board.is_game_over(claim_draw=True) and move_history:
            res = board.result(claim_draw=True)     # "1-0", "0-1", "1/2-1/2"
            reason = (
                "checkmate" if board.is_checkmate() else
                "stalemate" if board.is_stalemate() else
                "insufficient material" if board.is_insufficient_material() else
                "threefold repetition" if board.can_claim_threefold_repetition() else
                "fifty-move rule" if board.can_claim_fifty_moves() else
                "game over"
            )
            print("\nGame finished:", res, f"({reason}). Generating summary...\n")
            prompt = make_summary_prompt(res, reason, move_history)
            summary = clean_summary(safe_generate(prompt, max_new_tokens=320, temperature=0.85))
            print(summary)
            move_history.clear()
            print("\nThanks for playing. Goodbye! ♟️")
            break

        user = input("\nYour move (SAN/UCI) or 'exit' to quit with summary: ").strip()
        if not user:
            continue

        # Exit early -> summarize the current *incomplete* game and exit
        if user.lower() in ("exit", "quit"):
            if move_history:
                print("\nSession ending — someone quit mid-game. Generating summary...\n")
                res, reason = "*", ("White quit mid-game" if board.turn == chess.WHITE else "Black quit mid-game")
                prompt = make_summary_prompt(res, reason, move_history)
                summary = clean_summary(safe_generate(prompt, max_new_tokens=320, temperature=0.85))
                print(summary)
                move_history.clear()
            print("\nGoodbye!")
            break

        # Try SAN (strict), then UCI
        san_try = normalize_san(user)
        parsed = None
        try:
            parsed = board.parse_san(san_try)
        except Exception:
            # Try UCI
            try:
                move = chess.Move.from_uci(user.lower())
                if move not in board.legal_moves:
                    raise ValueError("Illegal UCI move")
                parsed = move
            except Exception:
                print("Invalid move. Use SAN (e.g., e4, Nf3, Bxb5+, O-O) or UCI (e2e4, g1f3).")
                continue

        # PRE-move data
        fen_before = board.fen()
        # Canonical SAN (with true +/# if applicable)
        san_canonical = board.san(parsed)

        # Heuristic verdict BEFORE pushing
        label, reasons = classify_move(board, parsed)
        verdict_line = humanize_commentary(san_canonical, label, reasons)
        recent = " ".join(move_history[-6:]) if move_history else ""

        # Execute move
        board.push(parsed)
        move_history.append(san_canonical)

        # LLM commentary (short + human)
        prompt = make_comment_prompt(fen_before, san_canonical, verdict_line, recent)
        llm = clean_commentary(safe_generate(prompt, max_new_tokens=56, temperature=0.75), max_sentences=2)

        # Print verdict + board
        print(f"\n{san_canonical}: {verdict_line} {llm}")
        print_board(board)

In [9]:
run_one_game()

Chess Commentary Chatbot (one game)
  -----------------
8 |♜|♞|♝|♛|♚|♝|♞|♜|
  -----------------
7 |♟|♟|♟|♟|♟|♟|♟|♟|
  -----------------
6 |⭘|⭘|⭘|⭘|⭘|⭘|⭘|⭘|
  -----------------
5 |⭘|⭘|⭘|⭘|⭘|⭘|⭘|⭘|
  -----------------
4 |⭘|⭘|⭘|⭘|⭘|⭘|⭘|⭘|
  -----------------
3 |⭘|⭘|⭘|⭘|⭘|⭘|⭘|⭘|
  -----------------
2 |♙|♙|♙|♙|♙|♙|♙|♙|
  -----------------
1 |♖|♘|♗|♕|♔|♗|♘|♖|
  -----------------
   a b c d e f g h



Your move (SAN/UCI) or 'exit' to quit with summary:  exit



Goodbye!


In [10]:
def summarize_moves(result_str: str, reason: str, san_moves: list[str], title: str):
    print("\n" + "="*80)
    print(f"{title} — Result: {result_str} ({reason})")
    print("="*80)
    prompt = make_summary_prompt(result_str, reason, san_moves)
    summary = clean_summary(safe_generate(prompt, max_new_tokens=320, temperature=0.85))
    print(summary)

# 1) Draw (by agreement): a calm Italian opening that peters out
draw_game = [
    "e4","e5","Nf3","Nc6","Bc4","Bc5","c3","Nf6","d3","d6","O-O","O-O",
    "Re1","a6","Bb3","Ba7","Nbd2","Be6","Nf1","Qd7","Be3","Bxe3","Nxe3","Rfe8",
    "h3","Rad8","Qe2","Ne7","Rad1","Ng6"
]

# 2) White win — Scholar’s Mate
white_win_game = ["e4","e5","Bc4","Nc6","Qh5","Nf6","Qxf7#"]

# 3) Black win — Fool’s Mate
black_win_game = ["f3","e5","g4","Qh4#"]

# 4) Someone quit mid-game — specify who based on side-to-move
quit_game = ["d4","d5","c4","e6","Nc3","Nf6","Bg5","Be7","e3"]

summarize_moves("1/2-1/2", "draw agreed", draw_game, "Demo Game 1: Draw")
summarize_moves("1-0", "checkmate", white_win_game, "Demo Game 2: White Win")
summarize_moves("0-1", "checkmate", black_win_game, "Demo Game 3: Black Win")
# For demo, infer quitter: build the final board and check whose turn it is
_tmp = chess.Board()
for mv in quit_game:
    _tmp.push_san(mv)
quitter_reason = "White quit mid-game" if _tmp.turn == chess.WHITE else "Black quit mid-game"
summarize_moves("*", quitter_reason, quit_game, "Demo Game 4: Someone Quit")


Demo Game 1: Draw — Result: 1/2-1/2 (draw agreed)
This game was a tense, drawn battle. The opening was built on a solid, positional structure, with White employing a strategic plan. The central control was established early, with e4 and e5 providing equal pressure. The development of the pieces was relatively balanced, but White’s initiative was somewhat hampered by Black’s solid pawn structure.  

The key turning point arrived in the mid-game, with the exchange of the knight for the bishop. This tactical exchange shifted the dynamic, forcing Black to react and revealing a potential weakness. The tactical idea here was to exploit the exposed pawn structure. However, the lack of a clear plan from White meant that it lacked decisive moves.  

Black's development was steady, but their lack of initiative hampered their overall position.  The opening's subtle nuances of positional play led to a draw. The endgame was a quiet affair, with both sides exchanging pawns, highlighting the strateg

In [11]:
def run_scripted_game(title: str, san_moves: list[str], result_str: str, reason: str, show_boards: bool = False):
    """
    Replays a SAN move list, prints commentary per move using the same heuristic+LLM pipeline,
    then prints a detailed post-game summary.
    """
    print("\n" + "="*100)
    print(f"{title} — Result: {result_str} ({reason})")
    print("="*100)

    board = chess.Board()
    move_history = []

    if show_boards:
        print_board(board)

    for idx, san in enumerate(san_moves, start=1):
        # Parse SAN safely (strip decorations, normalize)
        san_try = normalize_san(san)
        try:
            move = board.parse_san(san_try)
        except Exception as e:
            print(f"[Error parsing SAN '{san}' at ply {idx}: {e}]")
            break

        # Pre-move context
        fen_before = board.fen()
        canonical_san = board.san(move)  # will include real +/# if applicable
        label, reasons = classify_move(board, move)
        verdict_line = humanize_commentary(canonical_san, label, reasons)
        recent = " ".join(move_history[-6:]) if move_history else ""

        # Execute the move
        board.push(move)
        move_history.append(canonical_san)

        # LLM commentary
        prompt = make_comment_prompt(fen_before, canonical_san, verdict_line, recent)
        llm = clean_commentary(safe_generate(prompt, max_new_tokens=56, temperature=0.75), max_sentences=2)

        # Output one-liner per move
        move_no = (idx + 1) // 2
        ply_prefix = f"{move_no}..." if board.turn == chess.WHITE else f"{move_no}."
        print(f"{ply_prefix} {canonical_san}: {verdict_line} {llm}")

        if show_boards:
            print_board(board)

    # --- Post-game summary ---
    if result_str == "*":
        quitter = "White" if board.turn == chess.WHITE else "Black"
        reason_full = f"{quitter} quit mid-game"
    else:
        reason_full = reason

    prompt_sum = make_summary_prompt(result_str, reason_full, move_history)
    summary = clean_summary(safe_generate(prompt_sum, max_new_tokens=320, temperature=0.85))
    print("\n— Detailed Summary —")
    print(summary)

# Reuse the four demo games from Cell 11 (draw_game, white_win_game, black_win_game, quit_game)
run_scripted_game("Demo Game 1: Draw", draw_game, "1/2-1/2", "draw agreed", show_boards=False)
run_scripted_game("Demo Game 2: White Win", white_win_game, "1-0", "checkmate", show_boards=False)
run_scripted_game("Demo Game 3: Black Win", black_win_game, "0-1", "checkmate", show_boards=False)
run_scripted_game("Demo Game 4: Someone Quit", quit_game, "*", "someone quit mid-game", show_boards=False)


Demo Game 1: Draw — Result: 1/2-1/2 (draw agreed)
1. e4: [Good] Solid and purposeful. It improves central control. The move e4 is a classic opening, opening the center and establishing a strong pawn structure.
1... e5: [Inaccuracy] A bit soft. It improves central control. The move e5 immediately challenges the center, opening lines for the queen and bishop. It's a solid and reasonable development, though perhaps slightly passive, aiming for a quick, dynamic game.
2. Nf3: [Inaccuracy] A bit soft. It develops a minor piece. Nf3 is a standard opening move, aiming to control the center and develop a knight. It's a solid, but somewhat passive move, potentially leading to a less dynamic game.
2... Nc6: [Inaccuracy] A bit soft. It develops a minor piece. Developing the knight to c6 is a good move, controlling the center and preparing for a pawn advance. ```

**Explanation:**

The move Nc6 is a solid and principled development.
3. Bc4: [Inaccuracy] A bit soft. It develops a minor piece, impro

In [12]:
run_one_game()

Chess Commentary Chatbot (one game)
  -----------------
8 |♜|♞|♝|♛|♚|♝|♞|♜|
  -----------------
7 |♟|♟|♟|♟|♟|♟|♟|♟|
  -----------------
6 |⭘|⭘|⭘|⭘|⭘|⭘|⭘|⭘|
  -----------------
5 |⭘|⭘|⭘|⭘|⭘|⭘|⭘|⭘|
  -----------------
4 |⭘|⭘|⭘|⭘|⭘|⭘|⭘|⭘|
  -----------------
3 |⭘|⭘|⭘|⭘|⭘|⭘|⭘|⭘|
  -----------------
2 |♙|♙|♙|♙|♙|♙|♙|♙|
  -----------------
1 |♖|♘|♗|♕|♔|♗|♘|♖|
  -----------------
   a b c d e f g h



Your move (SAN/UCI) or 'exit' to quit with summary:  exit



Goodbye!


In [12]:
# ============================================================
# Evaluation: Commentary & Summary Quality (no NLTK needed)
# Metrics: TF-IDF Cosine, BLEU-lite, ROUGE-L, LengthRatio, LexicalRichness
# ============================================================

import re
import math
import numpy as np
from typing import List, Dict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def _tok(s: str) -> List[str]:
    # simple, robust tokenizer (lowercase, words & punctuation as tokens)
    return re.findall(r"\w+|[^\w\s]", s.lower(), flags=re.UNICODE)

def _ngrams(tokens: List[str], n: int):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)] if len(tokens) >= n else []

def _modified_precision(ref_tokens: List[str], gen_tokens: List[str], n: int) -> float:
    ref_ngrams = _ngrams(ref_tokens, n)
    gen_ngrams = _ngrams(gen_tokens, n)
    if not gen_ngrams:
        return 0.0
    ref_counts = {}
    for ng in ref_ngrams:
        ref_counts[ng] = ref_counts.get(ng, 0) + 1
    max_matched = 0
    used = {}
    for ng in gen_ngrams:
        if ref_counts.get(ng, 0) > used.get(ng, 0):
            used[ng] = used.get(ng, 0) + 1
            max_matched += 1
    # smoothing (add-one)
    return (max_matched + 1) / (len(gen_ngrams) + 1)

def _brevity_penalty(ref_len: int, gen_len: int) -> float:
    if gen_len == 0:
        return 0.0
    if gen_len > ref_len:
        return 1.0
    return math.exp(1 - ref_len / max(gen_len, 1))

def bleu_lite(ref: str, gen: str, max_n: int = 4) -> float:
    ref_tokens, gen_tokens = _tok(ref), _tok(gen)
    # geometric mean of precisions with smoothing
    precisions = []
    for n in range(1, max_n+1):
        precisions.append(_modified_precision(ref_tokens, gen_tokens, n))
    # avoid log(0) via smoothing done above
    log_p = sum(math.log(p) for p in precisions) / max_n
    geo_mean = math.exp(log_p)
    bp = _brevity_penalty(len(ref_tokens), len(gen_tokens))
    return float(geo_mean * bp)

def rouge_l(ref: str, gen: str) -> float:
    # ROUGE-L F-measure based on LCS length
    a, b = _tok(ref), _tok(gen)
    la, lb = len(a), len(b)
    if la == 0 or lb == 0:
        return 0.0
    # LCS DP
    dp = [[0]*(lb+1) for _ in range(la+1)]
    for i in range(1, la+1):
        ai = a[i-1]
        for j in range(1, lb+1):
            if ai == b[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    lcs = dp[la][lb]
    prec = lcs / lb if lb else 0.0
    rec = lcs / la if la else 0.0
    if prec + rec == 0:
        return 0.0
    beta2 = 1.2**2  # common ROUGE-L setting (favor recall slightly)
    f = (1 + beta2) * prec * rec / (rec + beta2 * prec + 1e-12)
    return float(f)

def tfidf_cosine_similarity(refs: List[str], gens: List[str]) -> List[float]:
    # joint vectorizer to keep vocabulary consistent
    corpus = refs + gens
    if len(corpus) == 0:
        return [0.0]*len(gens)
    vec = TfidfVectorizer(ngram_range=(1,2), min_df=1)
    X = vec.fit_transform(corpus)
    R = X[:len(refs)]
    G = X[len(refs):]
    sims = cosine_similarity(G, R)
    # pairwise (i-th gen with i-th ref); fallback to max if lengths mismatch
    out = []
    for i in range(len(gens)):
        if i < len(refs):
            out.append(float(sims[i, i]))
        else:
            out.append(float(sims[i].max() if sims.shape[1] > 0 else 0.0))
    return out

def lexical_richness(s: str) -> float:
    toks = _tok(s)
    return (len(set(toks)) / max(len(toks), 1)) if toks else 0.0

def evaluate_pairs(reference_texts: List[str], generated_texts: List[str]) -> List[Dict[str, float]]:
    # align lengths
    n = min(len(reference_texts), len(generated_texts))
    refs = reference_texts[:n]
    gens = generated_texts[:n]
    tfidf_sims = tfidf_cosine_similarity(refs, gens)
    results = []
    for i, (ref, gen) in enumerate(zip(refs, gens)):
        bleu = bleu_lite(ref, gen)
        rouge = rouge_l(ref, gen)
        length_ratio = len(_tok(gen)) / max(len(_tok(ref)), 1)
        lex_rich = lexical_richness(gen)
        results.append({
            "TFIDF_Cosine": round(tfidf_sims[i], 3),
            "BLEU_lite": round(bleu, 3),
            "ROUGE_L": round(rouge, 3),
            "LengthRatio": round(length_ratio, 2),
            "LexicalRichness": round(lex_rich, 2)
        })
    return results

def _print_results(block_name: str, results: List[Dict[str, float]]):
    if not results:
        print(f"[{block_name}] No pairs to evaluate.")
        return
    for i, r in enumerate(results, 1):
        print(f"--- {block_name} {i} ---")
        for k, v in r.items():
            print(f"{k}: {v}")
        print()
    # means
    keys = results[0].keys()
    means = {k: float(np.mean([r[k] for r in results])) for k in keys}
    print(f"=== {block_name} • Mean Scores ===")
    for k, v in means.items():
        print(f"{k}: {v:.3f}")
    print()

_glob = globals()

ref_comms = _glob.get("reference_commentaries")
gen_comms = _glob.get("generated_commentaries")
ref_sums  = _glob.get("reference_summaries")
gen_sums  = _glob.get("generated_summaries")

if not (isinstance(ref_comms, list) and isinstance(gen_comms, list)) and not (isinstance(ref_sums, list) and isinstance(gen_sums, list)):
    # Fallback demo so the cell runs
    ref_comms = ["The move Nf3 develops a knight and controls central squares."]
    gen_comms = ["Nf3 is a good developing move that helps control the center."]
    ref_sums  = ["White played harmoniously, improving activity and central control."]
    gen_sums  = ["White developed pieces well and gained central control."]

print(">>> Evaluating Commentary")
comm_results = evaluate_pairs(ref_comms, gen_comms)
_print_results("Commentary", comm_results)

print(">>> Evaluating Summary")
sum_results = evaluate_pairs(ref_sums, gen_sums)
_print_results("Summary", sum_results)


>>> Evaluating Commentary
--- Commentary 1 ---
TFIDF_Cosine: 0.092
BLEU_lite: 0.137
ROUGE_L: 0.263
LengthRatio: 1.09
LexicalRichness: 1.0

=== Commentary • Mean Scores ===
TFIDF_Cosine: 0.092
BLEU_lite: 0.137
ROUGE_L: 0.263
LengthRatio: 1.090
LexicalRichness: 1.000

>>> Evaluating Summary
--- Summary 1 ---
TFIDF_Cosine: 0.202
BLEU_lite: 0.26
ROUGE_L: 0.521
LengthRatio: 0.9
LexicalRichness: 1.0

=== Summary • Mean Scores ===
TFIDF_Cosine: 0.202
BLEU_lite: 0.260
ROUGE_L: 0.521
LengthRatio: 0.900
LexicalRichness: 1.000



In [14]:
# ============================================================
# 🔊 SpeechT5 TTS — auto-fix for Torch>=2.6, safetensors-first, sentencepiece,
#     optional hf_xet speedup, and robust fallback to pyttsx3
# Models: microsoft/speecht5_tts + microsoft/speecht5_hifigan
# ============================================================

import sys, subprocess, importlib, os, traceback, re

def _pip(*args):
    return subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

def _ensure(pkg_spec):
    try:
        importlib.import_module(pkg_spec.split(">=")[0].split("==")[0])
    except Exception:
        _pip(pkg_spec)

# --- core libs we need ---
for pkg in ["transformers>=4.41.0", "datasets", "soundfile", "sentencepiece", "huggingface_hub"]:
    _ensure(pkg)

# ---- Torch >= 2.6 fixer ----
def _version_tuple(v):
    return tuple(int(x) for x in re.findall(r"\d+", v)[:3]) if isinstance(v, str) else (0,0,0)

def _ensure_torch_min(min_ver="2.6.0"):
    try:
        import torch
        if _version_tuple(torch.__version__) >= _version_tuple(min_ver):
            return
        print(f"Upgrading torch to >= {min_ver} (current {torch.__version__}) …")
        # Try CUDA 12.1 first (works on most modern Windows GPUs with recent drivers)
        try:
            _pip("--upgrade", "torch", "--index-url", "https://download.pytorch.org/whl/cu121")
        except Exception:
            print("CUDA build install failed, trying CPU build…")
            _pip("--upgrade", "torch", "--index-url", "https://download.pytorch.org/whl/cpu")
    except Exception:
        # If torch not installed, install CPU as a safe default
        print("torch not present; installing CPU build…")
        _pip("torch", "--index-url", "https://download.pytorch.org/whl/cpu")
    # re-import to confirm
    import importlib as _il
    torch = _il.import_module("torch")
    print(f"torch ready: {torch.__version__}")

_ensure_torch_min("2.6.0")

# OPTIONAL: speed up downloads for repos using Xet storage
try:
    _pip("huggingface_hub[hf_xet]")
except Exception:
    pass  # purely optional

import numpy as np, soundfile as sf
from pathlib import Path
from datetime import datetime
from IPython.display import Audio, display

from datasets import load_dataset
from huggingface_hub.utils import HfHubHTTPError
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
import torch

MODEL_ID = "microsoft/speecht5_tts"
VOCODER_ID = "microsoft/speecht5_hifigan"
SAMPLE_RATE = 16000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUTPUT_DIR = Path("tts_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
session_dir = OUTPUT_DIR / f"session_{datetime.now().strftime('%Y%m%d-%H%M%S')}"
session_dir.mkdir(parents=True, exist_ok=True)

def _load_speecht5():
    """
    Load SpeechT5 + HiFiGAN without auth; prefer safetensors; return processor, model, vocoder, speaker_embeddings.
    """
    print(f"Loading {MODEL_ID} on {DEVICE} (unauthenticated, safetensors-first)…")
    # Neutralize any accidental HF token that previously caused 401s
    os.environ.pop("HUGGINGFACE_HUB_TOKEN", None)
    os.environ.pop("HF_TOKEN", None)
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

    # Prefer safetensors; if a model doesn't provide safetensors, transformers will fall back internally.
    processor = SpeechT5Processor.from_pretrained(MODEL_ID, token=None, local_files_only=False)
    model = SpeechT5ForTextToSpeech.from_pretrained(
        MODEL_ID, token=None, local_files_only=False, use_safetensors=True
    ).to(DEVICE)
    vocoder = SpeechT5HifiGan.from_pretrained(
        VOCODER_ID, token=None, local_files_only=False, use_safetensors=True
    ).to(DEVICE)

    # Neutral English speaker embedding from CMU Arctic xvectors
    ds = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
    speaker_embeddings = torch.tensor(ds[7306]["xvector"]).unsqueeze(0).to(DEVICE)

    print("✅ SpeechT5 + vocoder loaded.")
    return processor, model, vocoder, speaker_embeddings

tts_backend = {"name": None}

try:
    processor, model, vocoder, speaker_embeddings = _load_speecht5()

    @torch.inference_mode()
    def tts_fn(text: str, out_path: Path):
        if not text or not text.strip():
            return False
        inputs = processor(text=text, return_tensors="pt").to(DEVICE)
        speech = model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=vocoder)
        wav = speech.detach().cpu().numpy().squeeze()
        if np.max(np.abs(wav)) > 1e-6:
            wav = wav / np.max(np.abs(wav)) * 0.97
        sf.write(str(out_path), wav, SAMPLE_RATE)
        return True

    tts_backend["name"] = "speecht5"
    tts_backend["fn"] = tts_fn

except Exception as e:
    print("\n⚠️ SpeechT5 still failed. Falling back to offline pyttsx3…")
    traceback.print_exc(limit=2)

    try:
        import pyttsx3
    except ImportError:
        _pip("pyttsx3")
        import pyttsx3

    engine = pyttsx3.init()
    engine.setProperty("rate", 180)
    engine.setProperty("volume", 1.0)

    def tts_fn_pyttsx3(text: str, out_path: Path):
        if not text or not text.strip():
            return False
        engine.save_to_file(text, str(out_path))
        engine.runAndWait()
        return True

    tts_backend["name"] = "pyttsx3"
    tts_backend["fn"] = tts_fn_pyttsx3
    print("✅ Using pyttsx3 fallback (offline).")

# ---- Detect your variables in the notebook ----
glb = globals()
possible_commentary_vars = [
    "generated_commentaries", "commentary_log", "move_commentaries",
    "all_commentaries", "commentaries", "llm_commentaries"
]
commentaries = None
for name in possible_commentary_vars:
    if isinstance(glb.get(name), list) and all(isinstance(x, str) for x in glb[name]):
        commentaries = glb[name]
        print(f"Found commentaries in `{name}` ({len(commentaries)} items).")
        break
if commentaries is None:
    commentaries = [
        "Nf3 is a solid developing move controlling the center.",
        "Black counters with d5, opening the center early."
    ]
    print("No commentary list found. Using demo commentaries.")

possible_summary_vars = ["summary", "final_summary", "game_summary", "generated_summary"]
final_summary = None
for name in possible_summary_vars:
    if isinstance(glb.get(name), str) and glb[name].strip():
        final_summary = glb[name]
        print(f"Found summary in `{name}`.")
        break
if final_summary is None:
    final_summary = "White maintained a central advantage and converted in the endgame."
    print("No summary string found. Using demo summary.")

# ---- Generate audio files ----
from pathlib import Path
wav_paths = []
for i, text in enumerate(commentaries, start=1):
    path = session_dir / f"commentary_{i:03d}.wav"
    if tts_backend["fn"](text, path):
        wav_paths.append(path)
        print(f"[{tts_backend['name']}] Saved: {path}")

sum_path = session_dir / "final_summary.wav"
if tts_backend["fn"](final_summary, sum_path):
    wav_paths.append(sum_path)
    print(f"[{tts_backend['name']}] Saved summary: {sum_path}")

# ---- Inline playback ----
print("\n▶ Inline playback (first few files):")
for p in wav_paths[:8]:
    display(Audio(filename=str(p), autoplay=False))
print(f"\nAll audio files saved in: {session_dir.resolve()}")

Upgrading torch to >= 2.6.0 (current 2.5.1) …
torch ready: 2.5.1
Loading microsoft/speecht5_tts on cuda (unauthenticated, safetensors-first)…


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`



⚠️ SpeechT5 still failed. Falling back to offline pyttsx3…
✅ Using pyttsx3 fallback (offline).
No commentary list found. Using demo commentaries.
No summary string found. Using demo summary.


Traceback (most recent call last):
  File "C:\Users\Admin\AppData\Local\Temp\ipykernel_26752\2638568111.py", line 104, in <module>
    processor, model, vocoder, speaker_embeddings = _load_speecht5()
                                                    ^^^^^^^^^^^^^^^^
  File "C:\Users\Admin\AppData\Local\Temp\ipykernel_26752\2638568111.py", line 95, in _load_speecht5
    ds = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: Dataset scripts are no longer supported, but found cmu-arctic-xvectors.py


KeyboardInterrupt: 